# 01 — Basic EDA

**Amaç:** Şema, kolon rolleri ve target dengesini doğrulamak. Bu notebook model eğitmez.

In [ ]:
from pathlib import Path
import sys
ROOT = Path.cwd() if (Path.cwd() / "src").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT / "src"))
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from IPython.display import display
from kaggle_s6_e7.config import (FIGURES_DIR, TABLES_DIR, TARGET_COL, ID_COL,
    PLOT_SAMPLE_SIZE, RANDOM_STATE, ensure_report_dirs)
from kaggle_s6_e7.data import load_competition_data, infer_feature_columns, validate_schema
ensure_report_dirs()
train, test = load_competition_data()
validate_schema(train, test)
cat_cols, num_cols = infer_feature_columns(train)
plot_df = train.sample(min(PLOT_SAMPLE_SIZE, len(train)), random_state=RANDOM_STATE)
sns.set_theme(style="whitegrid")

## Dataset shape ve örnekler

In [ ]:
print("train:", train.shape, "test:", test.shape)
display(train.head())
display(test.head())
train.info()

## Kolon rolleri ve özet istatistikler

In [ ]:
print("Categorical:", cat_cols)
print("Numeric:", num_cols)
display(train[num_cols].describe(percentiles=[.01,.05,.5,.95,.99]).T)
for col in cat_cols:
    print(f"\n{col}")
    display(train[col].value_counts(dropna=False).to_frame("count"))

## Target dağılımı

In [ ]:
from kaggle_s6_e7.eda import target_summary
target_table = target_summary(train, TARGET_COL)
display(target_table)
target_table.to_csv(TABLES_DIR / "01_target_distribution.csv")
ax = target_table["rate"].plot.bar(figsize=(7,4), title="Target distribution")
ax.set_ylabel("rate"); plt.tight_layout()
plt.savefig(FIGURES_DIR / "01_target_distribution.png", dpi=150); plt.show()

## Karar notu

Balanced accuracy ana metric olduğundan gelecekte yalnız accuracy değil, **sınıf bazlı recall** da izlenmelidir. Doğrulanan yorumları `reports/eda_findings.md` içine aktar.